## Plotting gene expression in UMAP

This is a quick and dirty way to visualize the expression of a specific gene in the UMAP embedding of the TAL dataset. I wrote a custom function to plot the gene expression across a series of conditions.


In [ ]:
import os

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import numpy as np
import scanpy as sc
import scipy.sparse as sp

In [ ]:
def plot_gene_umap(
    adata,
    gene: str,
    threshold: float = 0.1,
    feature_name_col: str = "feature_name",
    subclass_col: str = "subclass.l2",
    umap_obsm_key: str = "X_umap",
    cmap: str = "tab10",
    background_color: str = "lightgrey",
    background_alpha: float = 0.5,
    point_size: float = 1.0,
    figsize: tuple = (6, 6),
    title_prefix: str = None,
    save: bool = False,
    save_dir: str = "figures/genes",
    filename: str = None,
    dpi: int = 300,
    ext: str = "png",
):
    """
    Highlight cells in a UMAP embedding where expression of a given gene exceeds a threshold.

    Args:
        adata (AnnData): Annotated data matrix containing:
          - `.var[feature_name_col]` with feature/gene names
          - `.X` (or a layer) with expression values
          - `.obs[subclass_col]` with a categorical annotation
          - `.obsm[umap_obsm_key]` with the 2D UMAP coordinates
        gene (str): Name of the gene/feature to highlight.
        threshold (float, optional): Minimum expression value for highlighting (default: 0.1).
        feature_name_col (str, optional): Column in `adata.var` storing gene names (default: `"feature_name"`).
        subclass_col (str, optional): Column in `adata.obs` for categorical grouping (default: `"subclass.l2"`).
        umap_obsm_key (str, optional): Key in `adata.obsm` for UMAP coords (default: `"X_umap"`).
        cmap (str, optional): Matplotlib colormap name for categories (default: `"tab10"`).
        background_color (str, optional): Color for all cells below threshold (default: `"lightgrey"`).
        background_alpha (float, optional): Alpha for background points (default: `0.5`).
        point_size (float, optional): Marker size for points (default: `1.0`).
        figsize (tuple, optional): Figure size (default: `(6, 6)`).
        title_prefix (str, optional): Text to prepend to the plot title (default: `None`).

    Returns:
        fig : matplotlib.figure.Figure
            The Figure object containing the plot.
        ax : matplotlib.axes.Axes
            The Axes object of the UMAP plot.
    """
    if gene not in adata.var[feature_name_col].values:
        raise ValueError(f"Gene '{gene}' not found in adata.var['{feature_name_col}'].")

    gene_idx = np.where(adata.var[feature_name_col] == gene)[0][0]
    X = adata[:, gene_idx].X
    expr = X.A.flatten() if sp.issparse(X) else X.flatten()
    mask_pos = expr > threshold

    umap = adata.obsm[umap_obsm_key]
    subcats = adata.obs[subclass_col].cat.categories
    palette = plt.get_cmap(cmap, len(subcats)).colors
    color_dict = dict(zip(subcats, palette))

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(
        umap[:, 0],
        umap[:, 1],
        c=background_color,
        s=point_size,
        alpha=background_alpha,
        label="Background",
    )

    for sub in subcats:
        idx = (adata.obs[subclass_col] == sub) & mask_pos
        ax.scatter(
            umap[idx, 0],
            umap[idx, 1],
            c=[color_dict[sub]],
            s=point_size,
            label=f"{sub} ({gene} > {threshold})",
        )

    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        markerscale=5,
    )
    title = f"{gene} > {threshold}"
    if title_prefix:
        title = f"{title_prefix}: {title}"
    ax.set_title(title)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

    ax.grid(False)
    ax.set_xlim(umap[:, 0].min() - 1, umap[:, 0].max() + 1)
    ax.set_ylim(umap[:, 1].min() - 1, umap[:, 1].max() + 1)
    ax.axis("off")

    if save:
        outdir = os.path.join(save_dir, gene)
        os.makedirs(outdir, exist_ok=True)
        if filename is None:
            safe_prefix = title_prefix.replace(" ", "_") if title_prefix else gene
            filename = f"{safe_prefix}_{gene}_{threshold}.{ext}"

        fpath = os.path.join(outdir, filename)
        fig.savefig(fpath, dpi=dpi, bbox_inches="tight")
        print(f"Saved plot to {fpath!r}")

    plt.show()
    return fig, ax

In [ ]:
adata = sc.read("data/kpmp_tal_filtered.h5ad")
adata

In [ ]:
diseases = {
    "acute kidney failure": "AKI",
    "chronic kidney disease": "CKD",
    "normal": "Reference",
}
thresholds = [0.1, 0.5, 1.0]

In [ ]:
gene = "UMOD"
threshold = 0.1

In [ ]:
for disease, acronym in diseases.items():
    adata_disease = adata[adata.obs["disease"] == disease].copy()
    print(f"Processing disease: {acronym}")
    for subclass in adata_disease.obs["merged_subclass"].cat.categories:
        adata_filtered_subclass = adata_disease[
            adata_disease.obs["merged_subclass"] == subclass
        ].copy()
        print(f"  Subclass: {subclass}")
        for threshold in [0.1, 0.5, 1.0]:
            fig, ax = plot_gene_umap(
                adata_filtered_subclass,
                gene=gene,
                threshold=threshold,
                title_prefix=f"{acronym} - {subclass}",
                save=True,
                filename=f"{acronym}-{subclass}_thr{threshold:.1f}.png",
            )
            plt.show()

In [ ]:
adata_filtered = adata[adata.obs["tissue"] == "kidney"].copy()
adata_filtered

In [ ]:
subclasses = adata_filtered.obs["merged_subclass"].cat.categories.tolist()
n_rows = len(subclasses)
n_cols = len(diseases)

fig = plt.figure(figsize=(6 * n_cols, 4 * n_rows))
gs = GridSpec(n_rows, n_cols, figure=fig)
fig.subplots_adjust(wspace=0.5, hspace=0.4)

_original_show = plt.show
plt.show = lambda *args, **kwargs: None

for i, subclass in enumerate(subclasses):
    for j, (disease, acronym) in enumerate(diseases.items()):
        ad = adata_filtered[
            (adata_filtered.obs["disease"] == disease)
            & (adata_filtered.obs["merged_subclass"] == subclass)
        ]

        small_fig, small_ax = plot_gene_umap(
            ad,
            gene=gene,
            threshold=threshold,
            title_prefix=f"{acronym} — {subclass}",
        )

        ax = fig.add_subplot(gs[i, j])
        for coll in small_ax.collections:
            ax.scatter(
                coll.get_offsets()[:, 0],
                coll.get_offsets()[:, 1],
                s=coll.get_sizes(),
                c=coll.get_facecolors(),
                alpha=coll.get_alpha(),
                label=coll.get_label(),
            )

        ax.set_title(small_ax.get_title())
        ax.set_xlim(small_ax.get_xlim())
        ax.set_ylim(small_ax.get_ylim())
        ax.set_aspect("equal", adjustable="box")
        ax.axis("off")

        if j == n_cols - 1:
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(
                handles=handles,
                labels=labels,
                bbox_to_anchor=(1.02, 1),
                loc="upper left",
                frameon=False,
                markerscale=5,
            )

        plt.close(small_fig)

plt.show = _original_show
fig.tight_layout()

fig.savefig("figures/tal_embedding_analysis.png", dpi=300, bbox_inches="tight")

plt.show()